# AI2 OLMo Agent Loop with Local Transformers

This standalone notebook runs an Ai2 OLMo model locally with `transformers`. It does not use Gemini, OpenAI, or hosted inference providers.

In [ ]:
%pip install -q "transformers>=4.57.0" torch accelerate

In [ ]:
import json
import os
import re
from datetime import date

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

AI2_MODEL = "allenai/Olmo-3-7B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN")
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float16 if device == "cuda" else torch.float32

print(f"Loading {AI2_MODEL} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(AI2_MODEL, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    AI2_MODEL,
    token=HF_TOKEN,
    torch_dtype=torch_dtype,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model = model.to(device)
model.eval()
print("Model ready.")

In [ ]:
TASK = """Compile a due-diligence snapshot for vendor 'Arion Data Systems'.
I need: founding year, approximate headcount, latest funding round (amount + date + lead investor), CEO name, reported ARR if available, and legal or compliance red flags. Include today's date."""

def get_company_profile(company_name):
    if "arion" in company_name.lower():
        return "Arion Data Systems | Founded: 2017 | Employees: ~340 | HQ: Amsterdam, NL | Sector: B2B SaaS"
    return f"No profile found for '{company_name}'"

def get_funding_history(company_name):
    if "arion" in company_name.lower():
        return "Series B: $28M (March 2023, Balderton Capital) | Series A: $9M (October 2020, Seedcamp) | Total raised: $38.5M"
    return f"No funding data for '{company_name}'"

def search_news(company_name, topic):
    key = f"{company_name} {topic}".lower()
    if "arion" not in key:
        return f"No news for '{company_name}' on '{topic}'"
    if any(term in key for term in ("ceo", "leadership", "founder")):
        return "CEO: Jonas Meier (co-founder). CTO: Priya Natarajan. No recent changes."
    if any(term in key for term in ("revenue", "arr", "financial", "growth")):
        return "Reported ARR EUR 12M (Q4 2023 investor deck). No audited public accounts."
    if any(term in key for term in ("legal", "gdpr", "compliance", "risk", "red flag")):
        return "Open GDPR complaint with Dutch DPA (Feb 2023). IP dispute settled in 2022."
    return "No significant news for that topic."

def get_current_date():
    return str(date.today())

TOOLS = {
    "get_company_profile": get_company_profile,
    "get_funding_history": get_funding_history,
    "search_news": search_news,
    "get_current_date": get_current_date,
}

In [ ]:
TOOL_GUIDE = """
Available tools:
- get_current_date: {}
- get_company_profile: {company_name: string}
- get_funding_history: {company_name: string}
- search_news: {company_name: string, topic: string}
"""
SYSTEM_PROMPT = f"""You are a corporate research assistant. Use tools before answering. Return exactly one JSON object and no Markdown. For a tool call use {{\"action\":\"tool\",\"name\":\"tool_name\",\"arguments\":{{...}}}}. For the final answer use {{\"action\":\"final\",\"answer\":\"...\"}}. Use one tool per turn.
{TOOL_GUIDE}"""

def generate_response(messages, max_new_tokens=500):
    prompt_messages = [{"role": "system", "content": SYSTEM_PROMPT}] + messages
    inputs = tokenizer.apply_chat_template(prompt_messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    inputs = {name: value.to(model.device) for name, value in inputs.items()}
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def parse_decision(text):
    candidate = text.strip()
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", candidate, re.DOTALL)
    if fenced:
        candidate = fenced.group(1)
    else:
        start, end = candidate.find("{"), candidate.rfind("}")
        if start >= 0 and end > start:
            candidate = candidate[start:end + 1]
    try:
        decision = json.loads(candidate)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Model returned non-JSON output: {text[:300]}") from exc
    if decision.get("action") == "final":
        return "final", str(decision.get("answer", ""))
    if decision.get("action") == "tool":
        name, arguments = decision.get("name"), decision.get("arguments", {})
        if name not in TOOLS or not isinstance(arguments, dict):
            raise ValueError(f"Invalid tool decision: {decision}")
        return "tool", name, arguments
    raise ValueError(f"Unknown action: {decision}")

In [ ]:
def run_ai2_agent(goal, max_steps=8):
    messages = [{"role": "user", "content": goal}]
    for step in range(1, max_steps + 1):
        print(f"\n[Step {step}] Calling local AI2 model...")
        raw_response = generate_response(messages)
        print(f"[RESPONSE] {raw_response[:300]}...")
        decision = parse_decision(raw_response)
        if decision[0] == "final":
            print("\nRECOMMENDATION:\n" + decision[1])
            return decision[1]
        _, name, arguments = decision
        result = TOOLS[name](**arguments)
        print(f"[TOOL] {name}({arguments})")
        print(f"[RESULT] {result}")
        messages.append({"role": "assistant", "content": raw_response})
        messages.append({"role": "user", "content": f"Tool result from {name}: {result}"})
    return "Max steps reached before the model returned a final answer."

In [ ]:
answer = run_ai2_agent(TASK, max_steps=8)